# Trotterized Ising Model

This notebook builds a Trotterized simulation circuit for the 1D transverse-field Ising model
and explores how circuit depth and gate count scale with system size.

In [ ]:
from qdk_pythonic import Circuit
import math

## The Ising Hamiltonian

The 1D transverse-field Ising Hamiltonian is:

$$H = -J \sum_{i} Z_i Z_{i+1} - h \sum_{i} X_i$$

A single first-order Trotter step decomposes into:

1. **ZZ interactions** for each adjacent pair: `CX(q[i], q[i+1])`, `Rz(angle, q[i+1])`, `CX(q[i], q[i+1])`
2. **Transverse field** on each qubit: `Rx(angle, q[i])`

## Define the Trotter Step

In [ ]:
def build_ising_trotter(
    n_qubits: int,
    n_steps: int = 1,
    j_coupling: float = 1.0,
    h_field: float = 0.5,
    dt: float = 0.1,
) -> Circuit:
    """Build a Trotterized 1D Ising model circuit.

    Args:
        n_qubits: Number of qubits in the 1D chain.
        n_steps: Number of Trotter steps.
        j_coupling: ZZ coupling strength.
        h_field: Transverse field strength.
        dt: Time step size.

    Returns:
        The constructed circuit.
    """
    circ = Circuit()
    q = circ.allocate(n_qubits, label="ising")

    zz_angle = -2 * j_coupling * dt
    rx_angle = -2 * h_field * dt

    for _ in range(n_steps):
        # ZZ interactions between adjacent qubits
        for i in range(n_qubits - 1):
            circ.cx(q[i], q[i + 1])
            circ.rz(zz_angle, q[i + 1])
            circ.cx(q[i], q[i + 1])

        # Transverse field: Rx on each qubit
        for i in range(n_qubits):
            circ.rx(rx_angle, q[i])

    return circ

## Small Example

Build a 4-qubit, single-step Trotter circuit and inspect it.

In [ ]:
circ_4 = build_ising_trotter(n_qubits=4, n_steps=1)
print(circ_4.draw())
print(f"Gate count: {circ_4.gate_count()}, Depth: {circ_4.depth()}")

## Scaling Analysis

Build Trotter circuits for n = 4, 8, and 12 qubits and compare the gate count and depth.

In [ ]:
sizes = [4, 8, 12]

print(f"{'Qubits':>8}  {'Gates':>8}  {'Depth':>8}")
print("-" * 30)

circuits = {}
for n in sizes:
    c = build_ising_trotter(n_qubits=n, n_steps=1)
    circuits[n] = c
    print(f"{n:>8}  {c.gate_count():>8}  {c.depth():>8}")

## Multiple Trotter Steps

Increasing the step count improves accuracy at the cost of more gates.

In [ ]:
n = 8
step_counts = [1, 5, 10, 20]

print(f"{'Steps':>8}  {'Gates':>8}  {'Depth':>8}")
print("-" * 30)

for steps in step_counts:
    c = build_ising_trotter(n_qubits=n, n_steps=steps)
    print(f"{steps:>8}  {c.gate_count():>8}  {c.depth():>8}")

## Generated Q# Code

Inspect the Q# output for the 4-qubit circuit.

In [ ]:
print(circuits[4].to_qsharp())

## Resource Estimation

> **Note:** The cells below require `qsharp>=1.25` (`pip install qsharp>=1.25`).

Estimate the physical resources needed for each system size.

In [ ]:
# Requires: pip install qsharp>=1.25
for n in sizes:
    result = circuits[n].estimate()
    print(f"n={n}: {result}")
    print()

## Batch Estimation Across Qubit Models

Compare resource costs for different qubit technologies.

In [ ]:
# Requires: pip install qsharp>=1.25
from qdk_pythonic.execution import estimate_circuit_batch

configs = [
    {"qubitParams": {"name": "qubit_gate_ns_e3"}},
    {"qubitParams": {"name": "qubit_maj_ns_e6"}},
]

batch_circuits = [circuits[n] for n in sizes]
results = estimate_circuit_batch(batch_circuits, configs)

for r in results:
    print(r)
    print()